In [1]:
pip install plotly

Note: you may need to restart the kernel to use updated packages.


In [2]:
# SLIDE 06
from pathlib import Path
import pandas as pd
import plotly.graph_objects as go

df = pd.read_csv("../data/df_model_delay.csv")  # must contain is_delayed (0/1)

counts = df["is_delayed"].value_counts().sort_index()
labels = ["On-time", "Delayed"]
values = [int(counts.get(0, 0)), int(counts.get(1, 0))]

fig = go.Figure(
    data=[
        go.Pie(
            labels=labels,
            values=values,
            hole=0.55,
            textinfo="percent",
            textfont=dict(size=18),  # bigger % labels
            hovertemplate="%{label}<br>%{value:,} events<br>%{percent}<extra></extra>",
            marker=dict(colors=["#38bdf8", "#e11d48"]),
        )
    ]
)

fig.update_layout(
    title=dict(text="Class balance: Delayed vs On-time (Delayed ≥ 1 min)", x=0.5),
    height=520,  # bigger chart canvas
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    font=dict(color="rgba(246,247,251,0.88)", size=16),  # bigger overall font
    margin=dict(l=20, r=180, t=70, b=20),  # extra space for the right legend
    showlegend=True,
    legend=dict(
        orientation="v",          # vertical legend on the right
        x=1.02, xanchor="left",
        y=0.5,  yanchor="middle",
        font=dict(size=18),
        bgcolor="rgba(0,0,0,0)"
    ),
)

out_path = Path("../outputs/figures/class_balance.html")
out_path.parent.mkdir(parents=True, exist_ok=True)
fig.write_html(out_path, full_html=True, include_plotlyjs=True)
print("Saved:", out_path)

Saved: ../outputs/figures/class_balance.html


In [3]:
# SLIDE 07
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go

MODEL_PATH = "../data/df_model_delay.csv"
RAW_PATH   = "../data/bus_delays_sample.csv"
OUT_DIR    = Path("../outputs/figures")

SPLIT = 15.0   # typical vs tail cutoff
CAP   = 75.0   # cap tail for readability (after ~75 it's extremely rare)

def load_delay_minutes():
    df = pd.read_csv(MODEL_PATH, low_memory=False)
    cols = {c.lower(): c for c in df.columns}

    if "delay_minutes" in cols:
        x = df[cols["delay_minutes"]].dropna().astype(float)
        return x, "df_model_delay.csv (delay_minutes)"

    if "planned_time" in cols and "real_time" in cols:
        planned = pd.to_datetime(df[cols["planned_time"]], errors="coerce")
        real = pd.to_datetime(df[cols["real_time"]], errors="coerce")
        x = (real - planned).dt.total_seconds() / 60.0
        return x.dropna().astype(float), "df_model_delay.csv (computed from timestamps)"

    df2 = pd.read_csv(RAW_PATH, low_memory=False)
    cols2 = {c.lower(): c for c in df2.columns}

    if "delay_minutes" in cols2:
        x = df2[cols2["delay_minutes"]].dropna().astype(float)
        return x, "bus_delays_sample.csv (delay_minutes)"

    if "planned_time" in cols2 and "real_time" in cols2:
        planned = pd.to_datetime(df2[cols2["planned_time"]], errors="coerce")
        real = pd.to_datetime(df2[cols2["real_time"]], errors="coerce")
        x = (real - planned).dt.total_seconds() / 60.0
        return x.dropna().astype(float), "bus_delays_sample.csv (computed from timestamps)"

    raise KeyError("Could not find delay_minutes or timestamps to compute it.")

x, source_used = load_delay_minutes()
x = x[x >= 0]  # keep it operationally simple (no early arrivals)

if len(x) > 2_000_000:
    x = x.sample(2_000_000, random_state=42)

p95 = float(np.percentile(x, 95))
p99 = float(np.percentile(x, 99))
xmax = float(np.max(x))

OUT_DIR.mkdir(parents=True, exist_ok=True)

# Make bars clearer + consistent
bar_line = dict(color="rgba(11,18,32,0.85)", width=0.8)
common_layout = dict(
    height=260,
    bargap=0.02,  # tighter bars
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    font=dict(color="rgba(246,247,251,0.88)", size=13),
    margin=dict(l=55, r=18, t=52, b=45),
    showlegend=False,
)

# ---------- Chart 1: Typical (0–15), 1-minute bins ----------
x_typ = x[(x >= 0) & (x <= SPLIT)]

fig1 = go.Figure()
fig1.add_trace(go.Histogram(
    x=x_typ,
    xbins=dict(start=0, end=SPLIT, size=1),  # <- fixes “separated bars”
    marker=dict(color="#38bdf8", line=bar_line),
    hovertemplate="Delay: %{x} min<br>Stop events: %{y:,}<extra></extra>"
))

# Percentile lines + labels (now explicit)
for val, tag, xshift in [(p95, "P95", -18), (p99, "P99", 18)]:
    if 0 <= val <= SPLIT:
        fig1.add_shape(
            type="line", x0=val, x1=val, y0=0, y1=1,
            xref="x", yref="paper",
            line=dict(color="#e11d48", width=2, dash="dot")
        )
        fig1.add_annotation(
            x=val, y=0.98, xref="x", yref="paper",
            text=f"{tag}={val:.1f}",
            showarrow=False,
            font=dict(size=12, color="rgba(246,247,251,0.92)"),
            xshift=xshift
        )

fig1.update_layout(
    title=dict(text="Typical delays (0–15 min)", x=0.5, font=dict(size=15)),
    **common_layout
)
fig1.update_xaxes(title="Delay (minutes)", range=[0, SPLIT], zeroline=False)
fig1.update_yaxes(title="Stop events", zeroline=False, tickformat="~s")

# ---------- Chart 2: Tail (>15), 1-minute bins, capped at 75+ ----------
x_tail = x[x > SPLIT]
x_tail_cap = x_tail.clip(upper=CAP)

fig2 = go.Figure()
fig2.add_trace(go.Histogram(
    x=x_tail_cap,
    xbins=dict(start=SPLIT, end=CAP, size=1),  # similar bar “feel”
    marker=dict(color="#e11d48", line=bar_line),
    hovertemplate="Delay (capped): %{x} min<br>Stop events: %{y:,}<extra></extra>"
))

fig2.add_shape(
    type="line", x0=CAP, x1=CAP, y0=0, y1=1, xref="x", yref="paper",
    line=dict(color="rgba(246,247,251,0.55)", width=2, dash="dash")
)
fig2.add_annotation(
    x=0.01, y=1.10, xref="paper", yref="paper",
    text=f"Capped at {int(CAP)}+ (max ≈ {xmax:.0f} min)",
    showarrow=False,
    font=dict(size=12, color="rgba(246,247,251,0.88)"),
    align="left"
)

fig2.update_layout(
    title=dict(text=f"Tail delays (> {int(SPLIT)} min)", x=0.5, font=dict(size=15)),
    **common_layout
)
fig2.update_xaxes(title=f"Delay (minutes, capped at {int(CAP)}+)", range=[SPLIT, CAP], zeroline=False)
fig2.update_yaxes(title="Stop events", zeroline=False, tickformat="~s")

config = dict(displayModeBar=False, responsive=True)

fig1.write_html(OUT_DIR / "delay_hist_typical.html", full_html=True, include_plotlyjs=True, config=config)
fig2.write_html(OUT_DIR / "delay_hist_tail.html", full_html=True, include_plotlyjs=True, config=config)

print("Saved charts to:", OUT_DIR)
print("Source used:", source_used)
print(f"SPLIT={SPLIT}, CAP={CAP}, P95={p95:.2f}, P99={p99:.2f}, max={xmax:.0f}, n={len(x):,}, tail_n={len(x_tail):,}")


Saved charts to: ../outputs/figures
Source used: bus_delays_sample.csv (delay_minutes)
SPLIT=15.0, CAP=75.0, P95=5.00, P99=9.00, max=1423, n=2,000,000, tail_n=5,871


In [4]:
# SLIDE 08 BBBBBB
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.graph_objects as go

DATA_PATH = "../data/df_model_delay.csv"
OUT_PATH  = Path("../outputs/figures/delayed_share_by_hour_B.html")

# Detect columns
cols = pd.read_csv(DATA_PATH, nrows=0).columns.tolist()
cols_l = {c.lower(): c for c in cols}

# Hour source
if "planned_hour" in cols_l:
    hour_col = cols_l["planned_hour"]
    usecols = [hour_col]
elif "planned_time" in cols_l:
    hour_col = cols_l["planned_time"]
    usecols = [hour_col]
else:
    raise KeyError("Need planned_hour or planned_time in df_model_delay.csv")

# Label source
label_cols = []
if "is_delayed" in cols_l:
    label_cols = [cols_l["is_delayed"]]
elif "delay_minutes" in cols_l:
    label_cols = [cols_l["delay_minutes"]]
elif "planned_time" in cols_l and "real_time" in cols_l:
    label_cols = [cols_l["planned_time"], cols_l["real_time"]]
else:
    raise KeyError("Need is_delayed, delay_minutes, or (planned_time + real_time) to create label")

usecols = list(dict.fromkeys(usecols + label_cols))
df = pd.read_csv(DATA_PATH, usecols=usecols, low_memory=False)

# Compute hour
if "planned_hour" in cols_l:
    hour = pd.to_numeric(df[cols_l["planned_hour"]], errors="coerce")
else:
    planned = pd.to_datetime(df[cols_l["planned_time"]], errors="coerce")
    hour = planned.dt.hour

# Compute is_delayed (Delayed >= 1 minute)
if "is_delayed" in cols_l and cols_l["is_delayed"] in df.columns:
    y = pd.to_numeric(df[cols_l["is_delayed"]], errors="coerce")
else:
    if "delay_minutes" in cols_l and cols_l["delay_minutes"] in df.columns:
        delay = pd.to_numeric(df[cols_l["delay_minutes"]], errors="coerce")
    else:
        planned = pd.to_datetime(df[cols_l["planned_time"]], errors="coerce")
        real = pd.to_datetime(df[cols_l["real_time"]], errors="coerce")
        delay = (real - planned).dt.total_seconds() / 60.0
    y = (delay >= 1).astype(int)

tmp = pd.DataFrame({"hour": hour, "is_delayed": y}).dropna()
tmp["hour"] = tmp["hour"].astype(int)
tmp = tmp[(tmp["hour"] >= 0) & (tmp["hour"] <= 23)]

rate = tmp.groupby("hour")["is_delayed"].mean().reindex(range(24))
cnt  = tmp.groupby("hour").size().reindex(range(24)).fillna(0).astype(int)

# --- Force-fill early hours if they are missing (use your verified values)
manual_rates = {0: 0.297, 1: 0.300, 2: 0.244}

for h, v in manual_rates.items():
    if pd.isna(rate.loc[h]) or cnt.loc[h] == 0:
        rate.loc[h] = v

# Optional: if those hours still have 0 events, mark them so hover shows "manual"
# (keeps the line/markers consistent and avoids “why 0 events?” confusion)
cnt_display = cnt.copy()
for h in manual_rates:
    if cnt_display.loc[h] == 0:
        cnt_display.loc[h] = np.nan  # will display as blank in hover if you use cnt_display


baseline = float(tmp["is_delayed"].mean())

# Top 3 hours (highest delayed share)
top3 = rate.sort_values(ascending=False).head(3)

x_hours = list(range(24))
y_rate = rate.values

fig = go.Figure()

# Main line with markers at every hour
fig.add_trace(go.Scatter(
    x=x_hours,
    y=y_rate,
    mode="lines+markers",
    line=dict(color="#38bdf8", width=3),
    marker=dict(size=8, color="#38bdf8", line=dict(color="rgba(11,18,32,0.85)", width=1)),
    customdata=cnt_display.values,
    hovertemplate="Hour: %{x}:00<br>Delayed share: %{y:.1%}<br>Stop events: %{customdata:,}<extra></extra>",
    name="Delayed share"
))

# Highlight top 3 points (same series, just overlay markers)
fig.add_trace(go.Scatter(
    x=[int(h) for h in top3.index],
    y=[float(v) for v in top3.values],
    mode="markers",
    marker=dict(size=13, color="#e11d48", line=dict(color="rgba(11,18,32,0.85)", width=1)),
    hovertemplate="Top hour: %{x}:00<br>Delayed share: %{y:.1%}<extra></extra>",
    name="Top hours"
))

# Baseline dashed line
fig.add_shape(
    type="line",
    x0=-0.5, x1=23.5, y0=baseline, y1=baseline,
    xref="x", yref="y",
    line=dict(color="rgba(246,247,251,0.60)", width=2, dash="dash")
)
fig.add_annotation(
    x=23.5, y=baseline, xref="x", yref="y",
    text=f"Baseline: {baseline:.1%}",
    showarrow=False,
    font=dict(size=12, color="rgba(246,247,251,0.85)"),
    xanchor="right", yanchor="bottom"
)

# Label top 3 (small, non-intrusive)
for h, v in top3.items():
    fig.add_annotation(
        x=int(h), y=float(v),
        xref="x", yref="y",
        text=f"{v:.1%}",
        showarrow=True, arrowhead=2, ax=0, ay=-28,
        font=dict(size=12, color="rgba(246,247,251,0.92)")
    )

fig.update_layout(
    title=None,
    height=520,
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    font=dict(color="rgba(246,247,251,0.88)", size=14),
    margin=dict(l=60, r=20, t=20, b=55),
    showlegend=True,
    legend=dict(
        orientation="h",
        x=0.5, xanchor="center",
        y=1.02, yanchor="bottom",
        font=dict(size=12)
    ),
)

# Show ALL hours 0..23
fig.update_xaxes(
    title="Planned hour of day",
    tickmode="array",
    tickvals=list(range(24)),
    ticktext=[f"{h:02d}" for h in range(24)],
    range=[-0.5, 23.5],
    showgrid=False,          # ✅ removes the busy vertical grid
    zeroline=False
)

fig.update_yaxes(
    title="Delayed share",
    tickformat=".0%",
    showgrid=True,           # keep only horizontal guides
    gridcolor="rgba(246,247,251,0.14)",
    zeroline=False
)

config = dict(displayModeBar=False, responsive=True)
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
fig.write_html(OUT_PATH, full_html=True, include_plotlyjs=True, config=config)

print("Saved:", OUT_PATH)
print("Baseline delayed share:", baseline)
print("Top 3 hours:", list(top3.index.astype(int)), list((top3.values*100).round(1)))


Saved: ../outputs/figures/delayed_share_by_hour_B.html
Baseline delayed share: 0.35979107480256095
Top 3 hours: [17, 9, 13] [np.float64(39.9), np.float64(39.1), np.float64(38.2)]


In [5]:
# SLIDE 09 — improved charts (y-axis to 50% + percent labels on bars)

from pathlib import Path
import pandas as pd
import numpy as np
import plotly.graph_objects as go

DATA_PATH = "../data/df_model_delay.csv"
OUT_DIR   = Path("../outputs/figures")

# Detect columns
cols = pd.read_csv(DATA_PATH, nrows=0).columns.tolist()
cols_l = {c.lower(): c for c in cols}

def compute_is_delayed(df):
    if "is_delayed" in cols_l and cols_l["is_delayed"] in df.columns:
        return pd.to_numeric(df[cols_l["is_delayed"]], errors="coerce")
    if "delay_minutes" in cols_l and cols_l["delay_minutes"] in df.columns:
        delay = pd.to_numeric(df[cols_l["delay_minutes"]], errors="coerce")
        return (delay >= 1).astype(int)
    planned = pd.to_datetime(df[cols_l["planned_time"]], errors="coerce")
    real = pd.to_datetime(df[cols_l["real_time"]], errors="coerce")
    delay = (real - planned).dt.total_seconds() / 60.0
    return (delay >= 1).astype(int)

usecols = []
if "planned_weekday" in cols_l: usecols.append(cols_l["planned_weekday"])
if "planned_hour" in cols_l: usecols.append(cols_l["planned_hour"])
if "time_band" in cols_l: usecols.append(cols_l["time_band"])
if "planned_time" in cols_l: usecols.append(cols_l["planned_time"])
if "is_delayed" in cols_l: usecols.append(cols_l["is_delayed"])
elif "delay_minutes" in cols_l: usecols.append(cols_l["delay_minutes"])
else:
    usecols += [cols_l["planned_time"], cols_l["real_time"]]

usecols = list(dict.fromkeys(usecols))
df = pd.read_csv(DATA_PATH, usecols=usecols, low_memory=False)

y = compute_is_delayed(df)
baseline = float(pd.Series(y).dropna().mean())

# ---- Weekday
if "planned_weekday" in cols_l and cols_l["planned_weekday"] in df.columns:
    wd = pd.to_numeric(df[cols_l["planned_weekday"]], errors="coerce")
else:
    planned = pd.to_datetime(df[cols_l["planned_time"]], errors="coerce")
    wd = planned.dt.weekday  # Monday=0

tmp_w = pd.DataFrame({"weekday": wd, "is_delayed": y}).dropna()
tmp_w["weekday"] = tmp_w["weekday"].astype(int)
tmp_w = tmp_w[(tmp_w["weekday"] >= 0) & (tmp_w["weekday"] <= 6)]

weekday_order = list(range(7))
weekday_names = ["Mon","Tue","Wed","Thu","Fri","Sat","Sun"]
w_rate = tmp_w.groupby("weekday")["is_delayed"].mean().reindex(weekday_order)
w_cnt  = tmp_w.groupby("weekday").size().reindex(weekday_order).fillna(0).astype(int)

# ---- Time band
if "time_band" in cols_l and cols_l["time_band"] in df.columns:
    tb = df[cols_l["time_band"]].astype(str)
else:
    if "planned_hour" in cols_l and cols_l["planned_hour"] in df.columns:
        hour = pd.to_numeric(df[cols_l["planned_hour"]], errors="coerce")
    else:
        planned = pd.to_datetime(df[cols_l["planned_time"]], errors="coerce")
        hour = planned.dt.hour

    def to_band(h):
        if pd.isna(h): return np.nan
        h = int(h)
        if 0 <= h <= 5:   return "night"
        if 6 <= h <= 9:   return "morning_peak"
        if 10 <= h <= 15: return "midday"
        if 16 <= h <= 19: return "evening_peak"
        if 20 <= h <= 23: return "late_evening"
        return np.nan

    tb = hour.map(to_band)

tmp_t = pd.DataFrame({"time_band": tb, "is_delayed": y}).dropna()
band_order = ["night","morning_peak","midday","evening_peak","late_evening"]
t_rate = tmp_t.groupby("time_band")["is_delayed"].mean().reindex(band_order)
t_cnt  = tmp_t.groupby("time_band").size().reindex(band_order).fillna(0).astype(int)

# ---- Common styling
bar_line = dict(color="rgba(11,18,32,0.85)", width=0.8)
config = dict(displayModeBar=False, responsive=True)

# ✅ Make axis look normal: always show 0%..50%
Y_MAX = 0.50

def apply_common_axes(fig):
    fig.update_yaxes(
        title="Delayed share",
        tickformat=".0%",
        range=[0, Y_MAX],
        showgrid=True,
        gridcolor="rgba(246,247,251,0.14)",
        zeroline=False
    )
    fig.update_xaxes(title=None, showgrid=False, zeroline=False)

def add_baseline(fig):
    fig.add_shape(
        type="line",
        x0=0, x1=1, xref="paper",
        y0=baseline, y1=baseline, yref="y",
        line=dict(color="rgba(246,247,251,0.60)", width=2, dash="dash")
    )

# ---- Chart 1: Weekday
fig_w = go.Figure()
fig_w.add_trace(go.Bar(
    x=weekday_names,
    y=w_rate.values,
    marker=dict(color="#38bdf8", line=bar_line),
    customdata=w_cnt.values,
    # ✅ label on top of each bar
    text=[f"{v*100:.1f}%" for v in w_rate.values],
    textposition="outside",
    cliponaxis=False,
    hovertemplate="%{x}<br>Delayed share: %{y:.1%}<br>Stop events: %{customdata:,}<extra></extra>"
))
add_baseline(fig_w)

fig_w.update_layout(
    title=dict(text="Delayed share by weekday", x=0.5, font=dict(size=15)),
    height=290,
    bargap=0.25,
    paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
    font=dict(color="rgba(246,247,251,0.88)", size=13),
    margin=dict(l=55, r=18, t=65, b=45),  # a bit more top space for labels
    showlegend=False,
)
apply_common_axes(fig_w)

# ---- Chart 2: Time bands
fig_t = go.Figure()
fig_t.add_trace(go.Bar(
    x=["night","morning peak","midday","evening peak","late evening"],
    y=t_rate.values,
    marker=dict(color="#e11d48", line=bar_line),
    customdata=t_cnt.values,
    # ✅ label on top of each bar
    text=[f"{v*100:.1f}%" for v in t_rate.values],
    textposition="outside",
    cliponaxis=False,
    hovertemplate="%{x}<br>Delayed share: %{y:.1%}<br>Stop events: %{customdata:,}<extra></extra>"
))
add_baseline(fig_t)

fig_t.update_layout(
    title=dict(text="Delayed share by time band", x=0.5, font=dict(size=15)),
    height=290,
    bargap=0.25,
    paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
    font=dict(color="rgba(246,247,251,0.88)", size=13),
    margin=dict(l=55, r=18, t=65, b=45),
    showlegend=False,
)
apply_common_axes(fig_t)

OUT_DIR.mkdir(parents=True, exist_ok=True)
fig_w.write_html(OUT_DIR / "delayed_share_by_weekday.html", full_html=True, include_plotlyjs=True, config=config)
fig_t.write_html(OUT_DIR / "delayed_share_by_timeband.html", full_html=True, include_plotlyjs=True, config=config)

print("Saved:", OUT_DIR / "delayed_share_by_weekday.html")
print("Saved:", OUT_DIR / "delayed_share_by_timeband.html")
print("Baseline delayed share:", baseline)

Saved: ../outputs/figures/delayed_share_by_weekday.html
Saved: ../outputs/figures/delayed_share_by_timeband.html
Baseline delayed share: 0.35979107480256095


In [6]:
# SLIDE 10 BBBBB
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.graph_objects as go

DATA_PATH = "../data/df_model_delay.csv"
OUT_DIR   = Path("../outputs/figures")

MIN_EVENTS = 10000
TOP_N = 10

# ---------- Load minimal columns ----------
cols = pd.read_csv(DATA_PATH, nrows=0).columns.tolist()
cols_l = {c.lower(): c for c in cols}

# Route key: prefer public-facing route_short_name
route_col = None
for cand in ["route_short_name", "route_id"]:
    if cand in cols_l:
        route_col = cols_l[cand]
        break
if route_col is None:
    raise KeyError("Need route_short_name or route_id in df_model_delay.csv")

usecols = [route_col]
if "is_delayed" in cols_l:
    usecols.append(cols_l["is_delayed"])
elif "delay_minutes" in cols_l:
    usecols.append(cols_l["delay_minutes"])
elif "planned_time" in cols_l and "real_time" in cols_l:
    usecols += [cols_l["planned_time"], cols_l["real_time"]]
else:
    raise KeyError("Need is_delayed, delay_minutes, or (planned_time + real_time)")

usecols = list(dict.fromkeys(usecols))
df = pd.read_csv(DATA_PATH, usecols=usecols, low_memory=False)

# ---------- Compute is_delayed safely ----------
if "is_delayed" in cols_l and cols_l["is_delayed"] in df.columns:
    y = pd.to_numeric(df[cols_l["is_delayed"]], errors="coerce")
else:
    if "delay_minutes" in cols_l and cols_l["delay_minutes"] in df.columns:
        delay = pd.to_numeric(df[cols_l["delay_minutes"]], errors="coerce")
    else:
        planned = pd.to_datetime(df[cols_l["planned_time"]], errors="coerce")
        real = pd.to_datetime(df[cols_l["real_time"]], errors="coerce")
        delay = (real - planned).dt.total_seconds() / 60.0
    y = (delay >= 1).astype(int)

tmp = pd.DataFrame({
    "route": df[route_col].astype(str),
    "is_delayed": y
}).dropna()

baseline = float(tmp["is_delayed"].mean())

# ---------- Aggregate + stability filter ----------
agg = tmp.groupby("route").agg(
    delayed_share=("is_delayed", "mean"),
    events=("is_delayed", "size")
).reset_index()

stable = agg[agg["events"] >= MIN_EVENTS].copy()
stable = stable.sort_values("delayed_share")

# If too few routes pass the filter, TOP_N must shrink to avoid overlap/confusion
if len(stable) < 2 * TOP_N:
    TOP_N = max(1, len(stable) // 2)
    print(f"Adjusted TOP_N to {TOP_N} because only {len(stable)} routes meet MIN_EVENTS={MIN_EVENTS}.")

# Best = lowest delayed share; Worst = highest delayed share
best = stable.head(TOP_N).sort_values("delayed_share", ascending=True).copy()
worst = stable.tail(TOP_N).sort_values("delayed_share", ascending=False).copy()

# ---------- SANITY CHECK (prevents inverted nonsense) ----------
best_max = float(best["delayed_share"].max()) if len(best) else np.nan
worst_min = float(worst["delayed_share"].min()) if len(worst) else np.nan

print("Baseline delayed share:", f"{baseline:.2%}")
print(f"Best routes range:  {float(best['delayed_share'].min()):.2%} .. {best_max:.2%}")
print(f"Worst routes range: {worst_min:.2%} .. {float(worst['delayed_share'].max()):.2%}")

if len(best) and len(worst) and worst_min < best_max:
    print("⚠️ Sanity check warning: worst_min < best_max. This usually means too few stable routes or filter too strict.")

bar_line = dict(color="rgba(11,18,32,0.85)", width=0.8)
config = dict(displayModeBar=False, responsive=True)

def make_vbar(df_plot, title, color_hex, y_max=None):
    # x labels (keep the same order as df_plot)
    x_labels = [f"Route {r}" for r in df_plot["route"].values]
    y_vals = df_plot["delayed_share"].values
    text_labels = [f"{v*100:.1f}%" for v in y_vals]

    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=x_labels,
        y=y_vals,
        marker=dict(color=color_hex, line=bar_line),
        customdata=df_plot["events"].values,
        text=text_labels,
        textposition="outside",
        textfont=dict(size=14, color="rgba(246,247,251,0.92)"),
        cliponaxis=False,
        hovertemplate="Route: %{x}<br>Delayed share: %{y:.1%}<br>Stop events: %{customdata:,}<extra></extra>",
    ))

    # Baseline line (now horizontal, because y-axis is delayed share)
    fig.add_shape(
        type="line",
        x0=0, x1=1, xref="paper",
        y0=baseline, y1=baseline, yref="y",
        line=dict(color="rgba(246,247,251,0.55)", width=2, dash="dash")
    )
    fig.add_annotation(
        x=0.5, xref="paper",
        y=baseline, yref="y",
        text=f"Baseline: {baseline:.1%}",
        showarrow=False,
        font=dict(size=12, color="rgba(246,247,251,0.82)"),
        xanchor="center",
        yanchor="bottom"
    )

    # Preserve category order (avoid Plotly sorting routes alphabetically)
    fig.update_xaxes(
        title=None,
        showgrid=False,
        zeroline=False,
        tickangle=-25,
        categoryorder="array",
        categoryarray=x_labels
    )

    # Give room above bars so the % labels are not cut
    ymax_auto = float(np.nanmax(y_vals)) if len(y_vals) else 0.4
    ymax = y_max if y_max is not None else max(0.45, ymax_auto * 1.25)

    fig.update_yaxes(
        title="Delayed share",
        tickformat=".0%",
        showgrid=True,
        gridcolor="rgba(246,247,251,0.14)",
        zeroline=False,
        range=[0, ymax]
    )

    fig.update_layout(
        title=dict(text=title, x=0.5, font=dict(size=15)),
        height=310,
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
        font=dict(color="rgba(246,247,251,0.88)", size=13),
        margin=dict(l=60, r=30, t=60, b=95),  # bigger bottom margin for x labels
        showlegend=False,
    )

    return fig


OUT_DIR.mkdir(parents=True, exist_ok=True)

# Titles requested
title_worst = f"Top {TOP_N} worst routes (min {MIN_EVENTS:,} events)"
title_best  = f"Top {TOP_N} best routes (min {MIN_EVENTS:,} events)"

# Best chart should go to 40%
fig_worst = make_vbar(worst, title_worst, "#e11d48")
fig_best  = make_vbar(best,  title_best,  "#38bdf8", y_max=0.40)

fig_worst.write_html(OUT_DIR / "worst_routes_B.html", full_html=True, include_plotlyjs=True, config=config)
fig_best.write_html(OUT_DIR / "best_routes_B.html",  full_html=True, include_plotlyjs=True, config=config)

print("Saved:", OUT_DIR / "worst_routes_B.html")
print("Saved:", OUT_DIR / "best_routes_B.html")
print("Stable routes included:", len(stable), " / total routes:", len(agg))

Baseline delayed share: 35.98%
Best routes range:  4.56% .. 12.46%
Worst routes range: 51.30% .. 58.61%
Saved: ../outputs/figures/worst_routes_B.html
Saved: ../outputs/figures/best_routes_B.html
Stable routes included: 160  / total routes: 568


In [7]:
# SLIDE 11 BBBBBB
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.graph_objects as go

DATA_PATH = "../data/df_model_delay.csv"
OUT_DIR   = Path("../outputs/figures")

MIN_EVENTS = 10000   # stops tend to be many; start higher for stability
TOP_N = 10

cols = pd.read_csv(DATA_PATH, nrows=0).columns.tolist()
cols_l = {c.lower(): c for c in cols}

# Stop identifier
stop_col = None
for cand in ["stop_id", "stop", "stop_code", "stop_name"]:
    if cand in cols_l:
        stop_col = cols_l[cand]
        break
if stop_col is None:
    raise KeyError("Need stop_id (or similar) in df_model_delay.csv")

usecols = [stop_col]
if "is_delayed" in cols_l:
    usecols.append(cols_l["is_delayed"])
elif "delay_minutes" in cols_l:
    usecols.append(cols_l["delay_minutes"])
elif "planned_time" in cols_l and "real_time" in cols_l:
    usecols += [cols_l["planned_time"], cols_l["real_time"]]
else:
    raise KeyError("Need is_delayed, delay_minutes, or (planned_time + real_time)")

usecols = list(dict.fromkeys(usecols))
df = pd.read_csv(DATA_PATH, usecols=usecols, low_memory=False)

# Compute is_delayed
if "is_delayed" in cols_l and cols_l["is_delayed"] in df.columns:
    y = pd.to_numeric(df[cols_l["is_delayed"]], errors="coerce")
else:
    if "delay_minutes" in cols_l and cols_l["delay_minutes"] in df.columns:
        delay = pd.to_numeric(df[cols_l["delay_minutes"]], errors="coerce")
    else:
        planned = pd.to_datetime(df[cols_l["planned_time"]], errors="coerce")
        real = pd.to_datetime(df[cols_l["real_time"]], errors="coerce")
        delay = (real - planned).dt.total_seconds() / 60.0
    y = (delay >= 1).astype(int)

tmp = pd.DataFrame({
    "stop": df[stop_col].astype(str),
    "is_delayed": y
}).dropna()

baseline = float(tmp["is_delayed"].mean())

agg = tmp.groupby("stop").agg(
    delayed_share=("is_delayed", "mean"),
    events=("is_delayed", "size")
).reset_index()

stable = agg[agg["events"] >= MIN_EVENTS].copy()
stable = stable.sort_values("delayed_share")

# Auto-adjust TOP_N if needed
if len(stable) < 2 * TOP_N:
    TOP_N = max(1, len(stable) // 2)
    print(f"Adjusted TOP_N to {TOP_N} because only {len(stable)} stops meet MIN_EVENTS={MIN_EVENTS}.")

best = stable.head(TOP_N).sort_values("delayed_share", ascending=True).copy()
worst = stable.tail(TOP_N).sort_values("delayed_share", ascending=False).copy()

# Sanity check
best_max = float(best["delayed_share"].max()) if len(best) else np.nan
worst_min = float(worst["delayed_share"].min()) if len(worst) else np.nan
print("Baseline delayed share:", f"{baseline:.2%}")
print(f"Best stops range:  {float(best['delayed_share'].min()):.2%} .. {best_max:.2%}")
print(f"Worst stops range: {worst_min:.2%} .. {float(worst['delayed_share'].max()):.2%}")

bar_line = dict(color="rgba(11,18,32,0.85)", width=0.8)
config = dict(displayModeBar=False, responsive=True)

def shorten_label(s, maxlen=22):
    return s if len(s) <= maxlen else s[:maxlen-1] + "…"

def make_hbar(df_plot, title, color_hex, x_max=None):
    y_labels = [f"Stop {shorten_label(s)}" for s in df_plot["stop"].values]
    text_labels = [f"{v*100:.1f}%" for v in df_plot["delayed_share"].values]

    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=df_plot["delayed_share"].values,
        y=y_labels,
        orientation="h",
        marker=dict(color=color_hex, line=bar_line),
        customdata=df_plot["events"].values,
        text=text_labels,
        textposition="outside",
        textfont=dict(size=14, color="rgba(246,247,251,0.92)"),
        cliponaxis=False,
        hovertemplate="Stop: %{y}<br>Delayed share: %{x:.1%}<br>Stop events: %{customdata:,}<extra></extra>",
    ))

    fig.add_shape(
        type="line",
        x0=baseline, x1=baseline, xref="x",
        y0=0, y1=1, yref="paper",
        line=dict(color="rgba(246,247,251,0.55)", width=2, dash="dash")
    )
    fig.add_annotation(
        x=baseline, y=1.06, xref="x", yref="paper",
        text=f"Baseline: {baseline:.1%}",
        showarrow=False,
        font=dict(size=12, color="rgba(246,247,251,0.82)"),
        xanchor="center"
    )

    fig.update_yaxes(autorange="reversed", showgrid=False, zeroline=False)

    fig.update_layout(
        title=dict(text=title, x=0.5, font=dict(size=15)),
        height=310,
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
        font=dict(color="rgba(246,247,251,0.88)", size=13),
        margin=dict(l=210, r=90, t=60, b=45),
        showlegend=False,
    )

    fig.update_xaxes(
        title="Delayed share",
        tickformat=".0%",
        showgrid=True,
        gridcolor="rgba(246,247,251,0.14)",
        zeroline=False,
        range=[0, x_max] if x_max is not None else None
    )

    return fig

OUT_DIR.mkdir(parents=True, exist_ok=True)

fig_worst = make_hbar(worst, f"Top {TOP_N} worst stops (min {MIN_EVENTS:,} events)", "#e11d48", x_max=0.75)
fig_best  = make_hbar(best,  f"Top {TOP_N} best stops (min {MIN_EVENTS:,} events)",  "#38bdf8", x_max=0.75)

fig_worst.write_html(OUT_DIR / "worst_stops_B.html", full_html=True, include_plotlyjs=True, config=config)
fig_best.write_html(OUT_DIR / "best_stops_B.html",  full_html=True, include_plotlyjs=True, config=config)

print("Saved:", OUT_DIR / "worst_stops_B.html")
print("Saved:", OUT_DIR / "best_stops_B.html")
print("Stable stops included:", len(stable), " / total stops:", len(agg))

Baseline delayed share: 35.98%
Best stops range:  1.40% .. 29.52%
Worst stops range: 55.56% .. 67.67%
Saved: ../outputs/figures/worst_stops_B.html
Saved: ../outputs/figures/best_stops_B.html
Stable stops included: 183  / total stops: 2494


In [8]:
# SLIDE 12
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.graph_objects as go

DATA_PATH = "../data/df_model_delay.csv"
OUT_PATH  = Path("../outputs/figures/delayed_share_by_operator.html")

# -------- helper: wrap long labels into 2–3 lines --------
def wrap_label(s, max_chars=22, max_lines=3):
    s = str(s).strip()
    if len(s) <= max_chars:
        return s
    words = s.split()
    lines, line = [], ""
    for w in words:
        if len(line) + (1 if line else 0) + len(w) <= max_chars:
            line = (line + " " + w).strip()
        else:
            lines.append(line)
            line = w
        if len(lines) >= max_lines - 1:
            break
    if line:
        lines.append(line)
    # if still words remaining, add ellipsis to last line
    used_words = " ".join(lines).split()
    if len(used_words) < len(words):
        lines[-1] = (lines[-1][:max(0, max_chars-1)] + "…").strip()
    return "<br>".join(lines[:max_lines])

# Detect columns
cols = pd.read_csv(DATA_PATH, nrows=0).columns.tolist()
cols_l = {c.lower(): c for c in cols}

if "operator" not in cols_l:
    raise KeyError("Column 'operator' not found in df_model_delay.csv")

usecols = [cols_l["operator"]]

# label source
if "is_delayed" in cols_l:
    usecols.append(cols_l["is_delayed"])
elif "delay_minutes" in cols_l:
    usecols.append(cols_l["delay_minutes"])
elif "planned_time" in cols_l and "real_time" in cols_l:
    usecols += [cols_l["planned_time"], cols_l["real_time"]]
else:
    raise KeyError("Need is_delayed, delay_minutes, or (planned_time + real_time)")

usecols = list(dict.fromkeys(usecols))
df = pd.read_csv(DATA_PATH, usecols=usecols, low_memory=False)

op = df[cols_l["operator"]].astype(str).fillna("Unknown")

# Compute is_delayed (Delayed >= 1 min)
if "is_delayed" in cols_l and cols_l["is_delayed"] in df.columns:
    y = pd.to_numeric(df[cols_l["is_delayed"]], errors="coerce")
else:
    if "delay_minutes" in cols_l and cols_l["delay_minutes"] in df.columns:
        delay = pd.to_numeric(df[cols_l["delay_minutes"]], errors="coerce")
    else:
        planned = pd.to_datetime(df[cols_l["planned_time"]], errors="coerce")
        real = pd.to_datetime(df[cols_l["real_time"]], errors="coerce")
        delay = (real - planned).dt.total_seconds() / 60.0
    y = (delay >= 1).astype(int)

tmp = pd.DataFrame({"operator": op, "is_delayed": y}).dropna()
baseline = float(tmp["is_delayed"].mean())

agg = tmp.groupby("operator").agg(
    delayed_share=("is_delayed", "mean"),
    events=("is_delayed", "size")
).reset_index()

# Sort: highest delayed share first (monitoring view)
agg = agg.sort_values("delayed_share", ascending=False)

# Wrapped labels for y-axis (more space for bars)
agg["operator_wrapped"] = agg["operator"].map(lambda s: wrap_label(s, max_chars=26, max_lines=3))

# Text labels at end of bars
agg["label"] = (agg["delayed_share"] * 100).round(1).astype(str) + "%"

bar_line = dict(color="rgba(11,18,32,0.85)", width=0.8)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=agg["delayed_share"],
    y=agg["operator_wrapped"],
    orientation="h",
    marker=dict(color="#a855f7", line=bar_line),  # purple accent (theme)
    text=agg["label"],
    textposition="outside",
    textfont=dict(size=14, color="rgba(246,247,251,0.92)"),
    cliponaxis=False,
    customdata=np.stack([agg["operator"].values, agg["events"].values], axis=1),
    hovertemplate="Operator: %{customdata[0]}<br>Delayed share: %{x:.1%}<br>Stop events: %{customdata[1]:,}<extra></extra>",
))

# Baseline vertical line + label
fig.add_shape(
    type="line",
    x0=baseline, x1=baseline, xref="x",
    y0=0, y1=1, yref="paper",
    line=dict(color="rgba(246,247,251,0.55)", width=2, dash="dash")
)
fig.add_annotation(
    x=baseline, y=1.08, xref="x", yref="paper",
    text=f"Baseline: {baseline:.1%}",
    showarrow=False,
    font=dict(size=12, color="rgba(246,247,251,0.82)"),
    xanchor="center"
)

# Ensure first item appears at top
fig.update_yaxes(autorange="reversed", showgrid=False, zeroline=False)

# ✅ Chart title added
fig.update_layout(
    title=dict(text="Delayed share by operator", x=0.5, font=dict(size=16)),
    height=660,
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    font=dict(color="rgba(246,247,251,0.88)", size=14),
    margin=dict(l=170, r=90, t=70, b=55),  # more room for x-axis ticks
    showlegend=False,
)

# ✅ X-axis ticks 0%..50%
fig.update_xaxes(
    title="Delayed share",
    tickformat=".0%",
    tickmode="array",
    tickvals=[0, 0.1, 0.2, 0.3, 0.4, 0.5],
    range=[0, 0.5],
    showgrid=True,
    gridcolor="rgba(246,247,251,0.14)",
    zeroline=False
)

config = dict(displayModeBar=False, responsive=True)
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
fig.write_html(OUT_PATH, full_html=True, include_plotlyjs=True, config=config)

print("Saved:", OUT_PATH)
print("Baseline delayed share:", baseline)
print(agg[["operator","delayed_share","events"]])

Saved: ../outputs/figures/delayed_share_by_operator.html
Baseline delayed share: 0.35979107480256095
                                            operator  delayed_share   events
4              Ville de Luxembourg - Service Autobus       0.394919  2701389
0             Régime Général des Transports Routiers       0.349164  3608687
3                                            Unknown       0.323981     6059
2                                               TICE       0.305746  1040174
1  Société Nationale des Chemins de Fer Luxembour...       0.074169      391


In [9]:
# SLIDE 13
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.graph_objects as go

MODEL_PATH = "../data/df_model_delay.csv"
RAW_PATH   = "../data/bus_delays_sample.csv"
OUT_PATH   = Path("../outputs/figures/delay_severity_by_timeband_boxplot.html")

CAP = 15.0              # ✅ lower cap so the “near-zero” mass is readable
MAX_OUTLIERS = 250      # ✅ sample outlier points per band (keeps HTML light)

def derive_time_band_from_hour(h):
    if pd.isna(h): return np.nan
    h = int(h)
    if 0 <= h <= 5:   return "night"
    if 6 <= h <= 9:   return "morning_peak"
    if 10 <= h <= 15: return "midday"
    if 16 <= h <= 19: return "evening_peak"
    if 20 <= h <= 23: return "late_evening"
    return np.nan

def load_delay_and_timeband():
    # Try model file first
    df = pd.read_csv(MODEL_PATH, low_memory=False)
    cols = {c.lower(): c for c in df.columns}

    # time band
    if "time_band" in cols:
        tb = df[cols["time_band"]].astype(str)
    else:
        if "planned_hour" in cols:
            hour = pd.to_numeric(df[cols["planned_hour"]], errors="coerce")
        elif "planned_time" in cols:
            planned = pd.to_datetime(df[cols["planned_time"]], errors="coerce")
            hour = planned.dt.hour
        else:
            hour = None
        tb = hour.map(derive_time_band_from_hour) if hour is not None else None

    # delay minutes
    if "delay_minutes" in cols:
        delay = pd.to_numeric(df[cols["delay_minutes"]], errors="coerce")
        return delay, tb, "df_model_delay.csv (delay_minutes)"
    elif "planned_time" in cols and "real_time" in cols:
        planned = pd.to_datetime(df[cols["planned_time"]], errors="coerce")
        real = pd.to_datetime(df[cols["real_time"]], errors="coerce")
        delay = (real - planned).dt.total_seconds() / 60.0
        return delay, tb, "df_model_delay.csv (computed from timestamps)"

    # Fallback to raw file
    df2 = pd.read_csv(RAW_PATH, low_memory=False)
    cols2 = {c.lower(): c for c in df2.columns}

    if "time_band" in cols2:
        tb2 = df2[cols2["time_band"]].astype(str)
    else:
        if "planned_hour" in cols2:
            hour2 = pd.to_numeric(df2[cols2["planned_hour"]], errors="coerce")
        elif "planned_time" in cols2:
            planned2 = pd.to_datetime(df2[cols2["planned_time"]], errors="coerce")
            hour2 = planned2.dt.hour
        else:
            raise KeyError("No time band or planned hour/time found to derive time bands.")
        tb2 = hour2.map(derive_time_band_from_hour)

    if "delay_minutes" in cols2:
        delay2 = pd.to_numeric(df2[cols2["delay_minutes"]], errors="coerce")
        return delay2, tb2, "bus_delays_sample.csv (delay_minutes)"
    elif "planned_time" in cols2 and "real_time" in cols2:
        planned2 = pd.to_datetime(df2[cols2["planned_time"]], errors="coerce")
        real2 = pd.to_datetime(df2[cols2["real_time"]], errors="coerce")
        delay2 = (real2 - planned2).dt.total_seconds() / 60.0
        return delay2, tb2, "bus_delays_sample.csv (computed from timestamps)"

    raise KeyError("Could not find delay_minutes or timestamps to compute delay minutes.")

delay, tb, source_used = load_delay_and_timeband()

tmp = pd.DataFrame({"time_band": tb, "delay_min": delay}).dropna()

# ✅ Focus on severity of delay events
tmp = tmp[tmp["delay_min"] >= 1]

order = ["night", "morning_peak", "midday", "evening_peak", "late_evening"]
labels_map = {
    "night": "night",
    "morning_peak": "morning peak",
    "midday": "midday",
    "evening_peak": "evening peak",
    "late_evening": "late evening"
}
tmp = tmp[tmp["time_band"].isin(order)]

true_max = float(tmp["delay_min"].max()) if len(tmp) else float("nan")

def box_stats(d: np.ndarray):
    q1, med, q3 = np.percentile(d, [25, 50, 75])
    iqr = q3 - q1
    lower_fence = q1 - 1.5 * iqr
    upper_fence = q3 + 1.5 * iqr
    # actual whiskers constrained by data
    lw = float(np.min(d[d >= lower_fence])) if np.any(d >= lower_fence) else float(np.min(d))
    uw = float(np.max(d[d <= upper_fence])) if np.any(d <= upper_fence) else float(np.max(d))
    return float(q1), float(med), float(q3), lw, uw

fig = go.Figure()

# Higher-contrast styling for dark theme
box_fill = "rgba(56,189,248,0.55)"     # brighter cyan
box_line = "rgba(56,189,248,0.95)"
outlier_color = "rgba(225,29,72,0.70)" # red points for outliers

for band in order:
    d = tmp.loc[tmp["time_band"] == band, "delay_min"].values
    if len(d) == 0:
        continue

    q1, med, q3, lw, uw = box_stats(d)

    # Add box using summary stats (lightweight, no massive point arrays)
    fig.add_trace(go.Box(
    x=[labels_map[band]],          # ✅ forces the box into the correct category
    q1=[q1], median=[med], q3=[q3],
    lowerfence=[lw], upperfence=[uw],
    boxpoints=False,
    fillcolor=box_fill,
    line=dict(color=box_line, width=2),
    whiskerwidth=0.6,
    hovertemplate=f"Time band: {labels_map[band]}<br>"
                  f"Median: {med:.1f} min<br>"
                  f"IQR: {q1:.1f}–{q3:.1f} min<extra></extra>"
))


    # Sample upper outliers and plot them as points (capped for display)
    out = d[d > uw]
    if len(out) > 0:
        if len(out) > MAX_OUTLIERS:
            out = np.random.choice(out, size=MAX_OUTLIERS, replace=False)
        out = np.clip(out, 0, CAP)
        fig.add_trace(go.Scatter(
            x=[labels_map[band]] * len(out),
            y=out,
            mode="markers",
            marker=dict(size=6, color=outlier_color, opacity=0.75),
            hovertemplate=f"Time band: {labels_map[band]}<br>Outlier delay (capped): %{{y:.1f}} min<extra></extra>",
            showlegend=False
        ))

fig.update_layout(
    title=dict(text="Delay severity by time band (delayed events only)", x=0.5, font=dict(size=16)),
    height=660,
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    font=dict(color="rgba(246,247,251,0.88)", size=14),
    margin=dict(l=70, r=30, t=70, b=105),  # ✅ extra bottom margin so x labels show
    showlegend=False
)

fig.update_xaxes(
    title="Time band",
    type="category",
    categoryorder="array",
    categoryarray=[labels_map[b] for b in order],
    showgrid=False,
    tickfont=dict(size=13),
    zeroline=False
)

fig.update_yaxes(
    title=f"Delay minutes (capped at {int(CAP)}+)",
    range=[0, CAP],
    showgrid=True,
    gridcolor="rgba(246,247,251,0.14)",
    zeroline=False
)

fig.add_annotation(
    x=0.01, y=1.07, xref="paper", yref="paper",
    text=f"Y-axis capped at {int(CAP)}+ min (true max ≈ {true_max:.0f} min). Red dots = sampled upper outliers. Source: {source_used}",
    showarrow=False,
    font=dict(size=12, color="rgba(246,247,251,0.75)"),
    align="left"
)

config = dict(displayModeBar=False, responsive=True)
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
fig.write_html(OUT_PATH, full_html=True, include_plotlyjs=True, config=config)

print("Saved:", OUT_PATH)
print("Source used:", source_used)
print("Rows (delayed only):", len(tmp), "True max delay:", true_max)

Saved: ../outputs/figures/delay_severity_by_timeband_boxplot.html
Source used: bus_delays_sample.csv (delay_minutes)
Rows (delayed only): 2646875 True max delay: 1423.0


In [10]:
# SLIDE 14
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.graph_objects as go

DATA_PATH = "../data/df_model_delay.csv"
OUT_PATH  = Path("../outputs/figures/correlation_heatmap.html")

df = pd.read_csv(DATA_PATH, low_memory=False)
cols_l = {c.lower(): c for c in df.columns}

# --- build target reference ---
if "is_delayed" in cols_l:
    df["__is_delayed__"] = pd.to_numeric(df[cols_l["is_delayed"]], errors="coerce")
elif "delay_minutes" in cols_l:
    delay = pd.to_numeric(df[cols_l["delay_minutes"]], errors="coerce")
    df["__is_delayed__"] = (delay >= 1).astype(int)
elif "planned_time" in cols_l and "real_time" in cols_l:
    planned = pd.to_datetime(df[cols_l["planned_time"]], errors="coerce")
    real = pd.to_datetime(df[cols_l["real_time"]], errors="coerce")
    delay = (real - planned).dt.total_seconds() / 60.0
    df["__is_delayed__"] = (delay >= 1).astype(int)
else:
    raise KeyError("Need is_delayed OR delay_minutes OR (planned_time + real_time) to create target reference.")

# --- choose numeric/binary candidates if present ---
candidates = [
    "planned_hour", "planned_weekday",
    "is_weekend",
    "rain", "precipitation", "is_rainy", "is_heavy_rain",
    "route_volume", "stop_volume",
]
selected = [cols_l[c] for c in candidates if c in cols_l]
selected.append("__is_delayed__")

X = df[selected].copy()
for c in X.columns:
    X[c] = pd.to_numeric(X[c], errors="coerce")
X = X.dropna(axis=1, how="all")

corr = X.corr(numeric_only=True)

# Pretty labels
rename = {}
for c in corr.columns:
    rename[c] = "is_delayed (target)" if c == "__is_delayed__" else c.replace("_", " ")
corr = corr.rename(index=rename, columns=rename)
labels = list(corr.columns)

# Theme-consistent diverging colors (red -> gray -> cyan)
theme_diverging = [
    [0.0, "rgba(225,29,72,0.85)"],
    [0.5, "rgba(246,247,251,0.12)"],
    [1.0, "rgba(56,189,248,0.85)"],
]

fig = go.Figure(data=go.Heatmap(
    z=corr.values,
    x=labels,
    y=labels,
    zmin=-1, zmax=1,
    colorscale=theme_diverging,
    # ✅ FIX: 'titleside' removed; older plotly rejects it
    colorbar=dict(title=dict(text="corr")),
    hovertemplate="x: %{x}<br>y: %{y}<br>corr: %{z:.2f}<extra></extra>",
))

# Optional in-cell numbers (only if matrix is small)
if len(labels) <= 12:
    ann = []
    for i, y in enumerate(labels):
        for j, x in enumerate(labels):
            ann.append(dict(
                x=x, y=y,
                text=f"{corr.values[i][j]:.2f}",
                showarrow=False,
                font=dict(size=11, color="rgba(246,247,251,0.85)")
            ))
    fig.update_layout(annotations=ann)

fig.update_layout(
    title=dict(text="Correlation heatmap (numeric/binary + target reference)", x=0.5, font=dict(size=16)),
    height=660,
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    font=dict(color="rgba(246,247,251,0.88)", size=13),
    margin=dict(l=120, r=40, t=70, b=140),
)

fig.update_xaxes(tickangle=-35, showgrid=False, zeroline=False)
fig.update_yaxes(showgrid=False, zeroline=False)

config = dict(displayModeBar=False, responsive=True)
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
fig.write_html(OUT_PATH, full_html=True, include_plotlyjs=True, config=config)

print("Saved:", OUT_PATH)
print("Columns used:", labels)

Saved: ../outputs/figures/correlation_heatmap.html
Columns used: ['planned hour', 'planned weekday', 'is weekend', 'rain', 'is rainy', 'is heavy rain', 'route volume', 'stop volume', 'is_delayed (target)']


In [11]:
# SLIDE 16
import os
import pandas as pd
import plotly.graph_objects as go

OUT_HTML = "../outputs/figures/model_comparison_table.html"

rows = [
    {"Model": "Random Forest (class_weight)", "Train note": "All data",           "F1": 0.6551, "Precision": 0.6752, "Recall": 0.6362, "ROC AUC": 0.8638},
    {"Model": "Random Forest (undersampled train)", "Train note": "Balanced train only", "F1": 0.6351, "Precision": 0.5940, "Recall": 0.6803, "ROC AUC": 0.7804},
    {"Model": "Gradient Boosting", "Train note": "1M subsample",                  "F1": 0.6028, "Precision": 0.5966, "Recall": 0.6090, "ROC AUC": 0.8072},
    {"Model": "Logistic Regression (baseline)", "Train note": "All data",         "F1": 0.4531, "Precision": 0.4801, "Recall": 0.4290, "ROC AUC": 0.6797},
]
df = pd.DataFrame(rows).sort_values("F1", ascending=False).reset_index(drop=True)

best_idx = int(df["F1"].idxmax())
n = len(df)

def fmt_pct(x): 
    return f"{x*100:.1f}%"

header_fill = "rgba(255,255,255,0.08)"
row_fill_default = "rgba(255,255,255,0.03)"
row_fill_best = "rgba(56,189,248,0.14)"  # subtle cyan highlight

fills = [(row_fill_best if i == best_idx else row_fill_default) for i in range(n)]

table = go.Table(
    columnwidth=[290, 150, 95, 105, 95, 95],
    header=dict(
        values=["Model", "Train note", "F1", "Precision", "Recall", "ROC AUC"],
        fill_color=header_fill,
        font=dict(color="white", size=16),
        align=["left","left","center","center","center","center"],
        height=44,
    ),
    cells=dict(
        values=[
            df["Model"].tolist(),
            df["Train note"].tolist(),
            [fmt_pct(x) for x in df["F1"]],
            [fmt_pct(x) for x in df["Precision"]],
            [fmt_pct(x) for x in df["Recall"]],
            [f"{x:.3f}" for x in df["ROC AUC"]],
        ],
        fill_color=[fills]*6,
        font=dict(color="white", size=15),
        align=["left","left","center","center","center","center"],
        height=44,
    )
)

fig = go.Figure(data=[table])

# IMPORTANT: no Plotly title inside the chart (your slide already has a title)
fig.update_layout(
    height=640,  # larger, fills more of the slide
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    margin=dict(l=10, r=10, t=10, b=10),
)

config = dict(displayModeBar=False, responsive=True)

# Write HTML with body margin=0 to remove iframe scrollbars
inner = fig.to_html(full_html=False, include_plotlyjs=True, config=config)
html = f"""<!doctype html>
<html>
<head>
  <meta charset="utf-8"/>
  <meta name="viewport" content="width=device-width, initial-scale=1"/>
  <style>
    html, body {{ height: 100%; margin: 0; padding: 0; overflow: hidden; background: transparent; }}
    #wrap {{ height: 100vh; width: 100vw; }}
    .plotly-graph-div {{ height: 100vh !important; width: 100vw !important; }}
  </style>
</head>
<body>
  <div id="wrap">{inner}</div>
</body>
</html>
"""

os.makedirs(os.path.dirname(OUT_HTML), exist_ok=True)
with open(OUT_HTML, "w", encoding="utf-8") as f:
    f.write(html)

print("Saved:", OUT_HTML)

Saved: ../outputs/figures/model_comparison_table.html


In [12]:
import pandas as pd

df = pd.read_csv("../data/bus_delays_sample.csv", usecols=["collected_at"])
df["collected_at"] = pd.to_datetime(df["collected_at"], errors="coerce")
print("Start:", df["collected_at"].min())
print("End:  ", df["collected_at"].max())

Start: 2025-07-25 20:13:32
End:   2025-09-16 16:47:31


In [13]:
# SLIDE 12 — Delayed share by operator (theme-consistent color)
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.graph_objects as go

DATA_PATH = "../data/df_model_delay.csv"
OUT_DIR   = Path("../outputs/figures")

# --- Read columns
cols = pd.read_csv(DATA_PATH, nrows=0).columns.tolist()
cols_l = {c.lower(): c for c in cols}

# --- Helper: delayed label
def compute_is_delayed(df):
    if "is_delayed" in cols_l and cols_l["is_delayed"] in df.columns:
        return pd.to_numeric(df[cols_l["is_delayed"]], errors="coerce")
    if "delay_minutes" in cols_l and cols_l["delay_minutes"] in df.columns:
        delay = pd.to_numeric(df[cols_l["delay_minutes"]], errors="coerce")
        return (delay >= 1).astype(int)
    # fallback timestamps if needed
    planned = pd.to_datetime(df[cols_l["planned_time"]], errors="coerce")
    real = pd.to_datetime(df[cols_l["real_time"]], errors="coerce")
    delay = (real - planned).dt.total_seconds() / 60.0
    return (delay >= 1).astype(int)

# --- Find operator column robustly
operator_candidates = [c for c in cols if "operator" in c.lower()]
if not operator_candidates:
    raise ValueError("No operator column found. Columns containing 'operator' were not detected.")
op_col = operator_candidates[0]  # pick the first match; change if you prefer another

# --- Load minimal data
usecols = [op_col]
if "is_delayed" in cols_l: usecols.append(cols_l["is_delayed"])
elif "delay_minutes" in cols_l: usecols.append(cols_l["delay_minutes"])
else: usecols += [cols_l["planned_time"], cols_l["real_time"]]

usecols = list(dict.fromkeys(usecols))
df = pd.read_csv(DATA_PATH, usecols=usecols, low_memory=False)

y = compute_is_delayed(df)
tmp = pd.DataFrame({"operator": df[op_col].astype(str), "is_delayed": y}).dropna()

# --- Filter tiny groups (optional but recommended)
cnt = tmp.groupby("operator").size().sort_values(ascending=False)
min_events = 2000  # adjust if you want more/less operators
keep_ops = cnt[cnt >= min_events].index
tmp = tmp[tmp["operator"].isin(keep_ops)]

# If too many operators remain, keep top 12 by volume
cnt2 = tmp.groupby("operator").size().sort_values(ascending=False)
top_ops = cnt2.head(12).index
tmp = tmp[tmp["operator"].isin(top_ops)]

rate = tmp.groupby("operator")["is_delayed"].mean()
cnt3 = tmp.groupby("operator").size()

# Sort by delayed share (or by volume if you prefer)
rate = rate.sort_values(ascending=False)
cnt3 = cnt3.reindex(rate.index)

# --- Styling
bar_line = dict(color="rgba(11,18,32,0.85)", width=0.8)
config = dict(displayModeBar=False, responsive=True)

# Main bar color: use your deck's light blue
BAR_COLOR = "#38bdf8"

fig = go.Figure()
fig.add_trace(go.Bar(
    x=rate.index.tolist(),
    y=rate.values,
    marker=dict(color=BAR_COLOR, line=bar_line),
    customdata=cnt3.values,
    text=[f"{v*100:.1f}%" for v in rate.values],
    textposition="outside",
    cliponaxis=False,
    hovertemplate="%{x}<br>Delayed share: %{y:.1%}<br>Stop events: %{customdata:,}<extra></extra>"
))

ymax = float(rate.max()) if len(rate) else 0.4
fig.update_layout(
    title=dict(text="Delayed share by operator", x=0.5, font=dict(size=15)),
    height=520,  # important: fits your Slide 12 container
    bargap=0.25,
    paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
    font=dict(color="rgba(246,247,251,0.88)", size=13),
    margin=dict(l=60, r=18, t=55, b=70),
    showlegend=False,
)
fig.update_yaxes(
    title="Delayed share",
    tickformat=".0%",
    range=[0, max(0.45, ymax * 1.25)],  # gives room for the % labels
    showgrid=True,
    gridcolor="rgba(246,247,251,0.14)",
    zeroline=False
)
fig.update_xaxes(title=None, showgrid=False, zeroline=False)

OUT_DIR.mkdir(parents=True, exist_ok=True)
fig.write_html(OUT_DIR / "delayed_share_by_operator.html", full_html=True, include_plotlyjs=True, config=config)
print("Saved:", OUT_DIR / "delayed_share_by_operator.html")
print("Operator column used:", op_col)

Saved: ../outputs/figures/delayed_share_by_operator.html
Operator column used: operator
